# Helios 02 — Visual Basemap: NAIP aerial imagery to raster PMTiles

![NAIP aerial to raster PMTiles](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/helios-02.png)

**Solar site-selection, step 2: the visual context.** Before scoring roofs, we want
to *see* them. This notebook stages **NAIP** (National Agriculture Imagery Program)
aerial imagery for San Francisco, reprojects it to web mercator with
`gbx_rst_to_webmercator`, slices it into an **XYZ tile pyramid** with
`gbx_rst_xyzpyramid`, packages the pyramid as raster **PMTiles** via
`gbx_pmtiles_agg`, and views it with `plot_pmtiles`. This aerial basemap sits behind
the building footprints from notebook 01.

> Runs on the **lightweight tier (Serverless)** by default.

In [ ]:
%run ./config_nb

In [ ]:
# Flip to True to fully rebuild this notebook's tables / re-download / re-tile.
FORCE_REBUILD = False

## 1. Stage NAIP aerial imagery (notebook helper)

NAIP is hosted as Cloud-Optimized GeoTIFFs in the public AWS Open Data registry
(`s3://naip-analytic`, public/requester-pays) and is also discoverable via STAC on
Planetary Computer (collection `naip`). NAIP does **not** get a module-level API in
GeoBrix — this is a **notebook-local helper** that fetches one SF tile and stages it
to the Volume (idempotent, FUSE-safe sequential copy). On Serverless it uses
rasterio's bundled GDAL (no `gdal_translate` CLI).

In [ ]:
import os
import shutil
from databricks.labs.gbx.sample import get_temp_dir          # node-local scratch helper

NAIP_DIR = f"{HELIOS_DIR}/naip"
os.makedirs(NAIP_DIR, exist_ok=True)   # Volume FUSE-safe; dbutils.fs.mkdirs would also work
NAIP_PATH = f"{NAIP_DIR}/sf_naip.tif"

# Offline guard: Planetary Computer STAC hits the network.  In Docker / CI without
# network access we fall back to a small single-band raster from the committed
# sample corpus so the reproject → pyramid → PMTiles → view chain can still run.
_OFFLINE = os.environ.get("GBX_HELIOS_OFFLINE", "").lower() in ("1", "true", "yes")

def _offline_fallback(dest):
    """Copy the nyc sentinel-2 (red-byte) sample as a local stand-in for NAIP.

    The sample is a single-band uint8 GeoTIFF in EPSG:32618 (UTM-18N), small
    enough for fast pipeline validation.  It is NOT NAIP, but it exercises the
    full reproject → XYZ pyramid → PMTiles path correctly.
    """
    _sample_root = os.environ.get(
        "GBX_SAMPLE_DATA_ROOT",
        "/Volumes/main/default/geobrix_samples/geobrix-examples",
    )
    src = f"{_sample_root}/nyc/sentinel2/nyc_sentinel2_red_byte.tif"
    shutil.copy(src, dest)
    print(f"... OFFLINE fallback: copied {src} -> {dest} ({os.path.getsize(dest):,} bytes)")

def stage_naip(dest=NAIP_PATH, bbox=SF_CITY_BBOX):
    """Stage one SF NAIP COG to the Volume via Planetary Computer STAC (idempotent)."""
    if os.path.exists(dest) and not FORCE_REBUILD:
        print(f"... NAIP already staged at {dest}")
        return dest
    if _OFFLINE:
        _offline_fallback(dest)
        return dest
    try:
        import planetary_computer as pc
        import pystac_client
        import rasterio
        cat = pystac_client.Client.open(
            "https://planetarycomputer.microsoft.com/api/stac/v1",
            modifier=pc.sign_inplace,
        )
        minx, miny, maxx, maxy = bbox
        item = next(cat.search(
            collections=["naip"],
            bbox=[minx, miny, maxx, maxy],
            limit=1,
        ).items())
        href = item.assets["image"].href
        tmp = get_temp_dir()
        local = tmp / "sf_naip.tif"
        with rasterio.open(href) as src:
            profile = {**src.profile, "driver": "GTiff"}
            win = src.window(minx, miny, maxx, maxy)   # crop to AOI — Volume-friendly size
            data = src.read(window=win)
            profile.update(
                width=data.shape[2],
                height=data.shape[1],
                transform=src.window_transform(win),
            )
            with rasterio.open(local, "w", **profile) as dst:
                dst.write(data)
        shutil.copy(str(local), dest)          # FUSE-safe sequential copy
        print(f"... staged NAIP -> {dest} ({os.path.getsize(dest):,} bytes)")
    except Exception as _e:
        print(f"... NAIP fetch failed ({type(_e).__name__}: {_e}); falling back to sample raster")
        _offline_fallback(dest)
    return dest

stage_naip()

## 2. Preview the source imagery

`plot_file` renders the staged GeoTIFF straight from the Volume (auto-decimation,
per-band percentile stretch). This is a static source preview (a raw source raster has
no tiled interactive form); the tiled PMTiles **product** is what the `INTERACTIVE_PLOTS`
toggle governs at the view step below.

In [ ]:
plot_file(NAIP_PATH, fig_w=8, fig_h=6)

## 3. Load the imagery into a typed tile

We read the GeoTIFF bytes with the `binaryFile` reader and build a typed `tile`
struct via `rst_fromcontent` — the temp-file-free path that avoids executor races.

Loading a **single file** produces exactly one source tile. One source → one set of
non-overlapping XYZ tiles — there can be no `(z, x, y)` duplicates from this step.
If you later want to mosaic multiple NAIP scenes into one seamless image before
pyramiding, stage them into separate rows and merge (e.g. `gbx_rst_merge_agg`) before
the `gbx_rst_to_webmercator` step — `gbx_pmtiles_agg` keeps first-wins for raster
tiles, so overlapping sources would silently lose all but the first.

In [ ]:
naip = (
    spark.read.format("binaryFile").load(NAIP_PATH)
         .select(rx.rst_fromcontent(F.col("content"), F.lit("GTiff")).alias("tile"))
)
print(f"... loaded {naip.count()} source tile(s)")

## 4. Reproject to web mercator

XYZ / PMTiles tiles live in web mercator (EPSG:3857). `gbx_rst_to_webmercator`
reprojects the tile so the pyramid aligns to the slippy-map grid.

In [ ]:
naip_3857 = naip.select(rx.rst_to_webmercator("tile").alias("tile"))

## 5. Build the XYZ tile pyramid

`gbx_rst_xyzpyramid` slices the reprojected raster into a pyramid of slippy-map PNG
tiles across a zoom range, emitting `(z, x, y, bytes)` rows. We pick a
city-scale zoom range (z12–z16) to match notebook 01.

The UDTF is called via `LATERAL` and emits flat columns `z, x, y, bytes` directly —
no `LATERAL VIEW` or `F.explode` needed.

In [ ]:
naip_3857.createOrReplaceTempView("sf_naip_tile")

# gbx_rst_xyzpyramid signature: gbx_rst_xyzpyramid(tile, min_z, max_z [, format [, size [, resampling]]])
# Light-tier UDTF output columns (flat, no explode): z INTEGER, x INTEGER, y INTEGER, bytes BINARY
# Use LATERAL (not LATERAL VIEW) — the UDTF emits rows directly.
xyz = spark.sql("""
    SELECT t.z, t.x, t.y, t.bytes AS png
    FROM sf_naip_tile,
         LATERAL gbx_rst_xyzpyramid(tile, 12, 16) t
""")

sf_xyz = finalize_delta(xyz, "sf_naip_xyz_tiles")
print(f"... {sf_xyz.count():,} (z,x,y) raster tiles across z12-z16")

## 6. Package as raster PMTiles

Same `gbx_pmtiles_agg` aggregate as notebook 01 — it auto-detects PNG tiles and
writes a **raster** PMTiles archive.

For raster tiles `gbx_pmtiles_agg` uses **first-wins** per `(z, x, y)` — raster
images cannot be merged the way MVT features can. Since our pyramid comes from a
single merged source, every `(z, x, y)` is unique and no tiles are dropped.

In [ ]:
# gbx_pmtiles_agg signature: gbx_pmtiles_agg(bytes, z, x, y [, metadata_json])
# For raster tiles (PNG/JPEG/WEBP), gbx_pmtiles_agg keeps first-wins per (z,x,y).
# One source tile → one non-overlapping pyramid → no tiles lost.
archive_row = (
    sf_xyz.groupBy(F.lit(1).alias("_g"))
          .agg(F.expr("gbx_pmtiles_agg(png, z, x, y)").alias("archive"))
          .select("archive")
          .collect()[0]
)
TILES_DIR = f"{HELIOS_DIR}/tiles"
os.makedirs(TILES_DIR, exist_ok=True)   # Volume FUSE-safe sequential write from driver
NAIP_PMTILES = f"{TILES_DIR}/sf_naip.pmtiles"
with open(NAIP_PMTILES, "wb") as f:
    f.write(archive_row["archive"])
print(f"... wrote {NAIP_PMTILES} ({os.path.getsize(NAIP_PMTILES):,} bytes)")

## 7. View the raster PMTiles inline

`show_pmtiles` renders through the `INTERACTIVE_PLOTS` toggle — a static image by
default (GitHub/docs-renderable), or an interactive MapLibre raster layer when
`INTERACTIVE_PLOTS = True`.

In [ ]:
show_pmtiles(NAIP_PMTILES)

## What we built

- `sf_naip_xyz_tiles` (Delta) — one row per `(z, x, y)` PNG tile.
- `sf_naip.pmtiles` (Volume) — a self-contained aerial basemap.

Next: **notebook 03** adds the analytical layer — terrain slope and aspect from a
USGS 3DEP DEM, the inputs to a solar suitability score.